# hw3 - ensembles

In [167]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor

from rich import box
from rich.console import Console
from rich.table import Table
from tqdm import tqdm

## 1 Подготовка данных

Загрузите и предобработайте данные (по своему усмотрению) из hw1

In [168]:
data = pd.read_csv("train_features_with_answers.csv")
data.shape

(454, 31)

In [169]:
# Смотрим, в каких столбцах есть пропуски
if data.isnull().any().any():
    for col in data:
        passes = data[col].isnull().sum()
        if passes > 0:
            print(f"В {col} есть {passes} пропусков")

В age есть 28 пропусков
В address есть 5 пропусков


In [170]:
# Заменяем пропуски на медиану
age_median = data["age"].dropna().median()
data["age"] = data["age"].fillna(age_median)

# Заменяем пропуски на моду
adress_mode = data["address"].dropna().mode()[0]
data["address"] = data["address"].fillna(adress_mode)

In [171]:
# Закодируем категориальные данные
def encode_dataset(df, column):
    df_encoded = df.copy()

    if column in df_encoded.columns:
        target = df_encoded[column]
        df_encoded = df_encoded.drop(columns=[column])

    categorical_cols = df_encoded.select_dtypes(include=["object"]).columns

    df_encoded = pd.get_dummies(df_encoded, columns=categorical_cols, drop_first=True)

    if column in df.columns:
        df_encoded[column] = target

    return df_encoded

In [172]:
data = encode_dataset(data, "G3")
data.shape

(454, 44)

## 2 Обоснуйте выбор слабых (базовых) алгоритмов

In [173]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

base_models = {
    "linear": LinearRegression(),
    "ridge": Ridge(alpha=1.0),
    "lasso": Lasso(alpha=0.1),
    "decision_tree": DecisionTreeClassifier(
        max_depth=5, min_samples_split=20, random_state=42
    ),
    "knn": KNeighborsClassifier(n_neighbors=7),
}

### Обоснование выбора моделей:

**1. Линейная регрессия**
- Способность эффективно работать с большим количеством бинарных признаков, полученных после One-Hot Encoding
- Низкие вычислительные затраты при обучении и прогнозировании
- Прозрачная интерпретация влияния каждого признака на целевую переменную (успеваемость)

**2. Метод k-ближайших соседей**
- Возможность идентификации студентов со схожими характеристиками
- Учет локальных зависимостей и специфических закономерностей в данных
- Адаптивность к сложным нелинейным взаимосвязям между признаками

**3. Дерево решений**
- Выявление сложных нелинейных зависимостей между признаками и целевой переменной
- Автоматический отбор наиболее значимых признаков
- Устойчивость к выбросам и отсутствие требований к нормализации данных

## 3 Постройте решение на основе подхода Blending

Правила:
- Нужно использовать вероятности
- Предложите что-то лучше, чем брать среднее от предсказаний моделей (оценивать уверенность алгоритмов, точности и т.д.)
- Заставьте базовые алгоритмы быть некорелированными
- Добавьте рандома (например, стройте ваши алгоритмы на разных выборках, по разному предобрабатывайте данные или применяйте для разных признаков соответствующие алгоритмы ... )
- Проявите смекалку
- Цель: метрика MSE на тесте меньше 10

## Создаем выборки

In [174]:
def create_diverse_samples(features, target, random_seed=17):
    np.random.seed(random_seed)
    sample_collection = {}

    # 1. Основная тренировочная выборка
    # Стандартное разделение на обучающую и тестовую выборки в пропорции 80/20
    X_train, _, y_train, _ = train_test_split(
        features, target, train_size=0.8, random_state=random_seed
    )
    sample_collection["full_70"] = (X_train, y_train)

    # 2. Произвольная подвыборка
    # Формирование случайной подвыборки объемом 70% от исходного набора данных
    random_indices = np.random.choice(
        len(features), size=int(0.7 * len(features)), replace=False
    )
    sample_collection["random_60"] = (
        features.iloc[random_indices],
        target.iloc[random_indices],
    )

    # 3. Сбалансированная по гендерному признаку выборка
    # Формирование выборки с равным количеством представителей каждого пола
    # для исключения доминирования одной категории
    if "sex_M" in features.columns:
        male_indices = features[features["sex_M"] == 1].index
        female_indices = features[features["sex_M"] == 0].index

        min_count = min(len(male_indices), len(female_indices))
        if min_count > 0:
            balanced_indices = np.concatenate(
                [
                    np.random.choice(male_indices, min_count, replace=False),
                    np.random.choice(female_indices, min_count, replace=False),
                ]
            )
            sample_collection["balanced_sex"] = (
                features.loc[balanced_indices],
                target.loc[balanced_indices],
            )

    # 4. Сбалансированная по уровню образования выборка
    # Формирование выборки с равным представительством трех наиболее распространенных
    # уровней материнского образования для минимизации влияния доминирующей категории
    if "Medu" in features.columns:
        top_education_categories = features["Medu"].value_counts().head(3).index
        education_balanced_indices = []

        for education_category in top_education_categories:
            category_indices = features[features["Medu"] == education_category].index
            if len(category_indices) > 0:
                category_sample_size = min(80, len(category_indices))
                education_balanced_indices.extend(
                    np.random.choice(
                        category_indices, category_sample_size, replace=False
                    )
                )

        if education_balanced_indices:
            sample_collection["balanced_education"] = (
                features.loc[education_balanced_indices],
                target.loc[education_balanced_indices],
            )

    return sample_collection


X = data.drop(columns=["G3"])
y = data["G3"]
generated_samples = create_diverse_samples(X, y)
print("\n СФОРМИРОВАННЫЕ ВЫБОРКИ:")
print("─" * 50)
for sample_name, (X_sampled, y_sampled) in generated_samples.items():
    print(
        f"  ▪ {sample_name:<18} → {X_sampled.shape[0]:>4} записей, {X_sampled.shape[1]:>2} признаков"
    )
print("─" * 50)


 СФОРМИРОВАННЫЕ ВЫБОРКИ:
──────────────────────────────────────────────────
  ▪ full_70            →  363 записей, 43 признаков
  ▪ random_60          →  317 записей, 43 признаков
  ▪ balanced_sex       →  378 записей, 43 признаков
  ▪ balanced_education →  240 записей, 43 признаков
──────────────────────────────────────────────────


In [175]:
# Возвращает метрики для регрессионной модели
# Показывает уверенность модели
def calculate_model_confidence_regression(y_true, y_pred, model_name=""):
    metrics = {}

    # 1. R² score - основная метрика качества
    r2 = r2_score(y_true, y_pred)
    metrics["r2"] = max(0, r2)

    # 2. Средняя абсолютная ошибка (MAE)
    mae = mean_absolute_error(y_true, y_pred)
    mae_confidence = max(0, 1 - mae / 5.0)
    metrics["mae"] = mae

    # 3. Среднеквадратичная ошибка (MSE)
    mse = mean_squared_error(y_true, y_pred)
    mse_confidence = max(0, 1 - mse / 25.0)
    metrics["mse"] = mse

    # 4. Доля предсказаний в пределах допустимой ошибки (+-1 балл)
    within_tolerance = np.mean(np.abs(y_true - y_pred) <= 1.0)
    metrics["precision_confidence"] = within_tolerance

    # 5. Комбинированная уверенность (простое среднее)
    combined_confidence = np.mean(
        [metrics["r2"], mae_confidence, mse_confidence, within_tolerance]
    )
    metrics["combined_confidence"] = combined_confidence

    return metrics

## BlendingModel

In [176]:
class BlendingModel:
    def __init__(self, base_models, random_state=42):
        self.base_models = base_models
        self.model_confidences = {}
        self.random_state = random_state

    def fit(self, X_train, y_train):
        diverse_samples = create_diverse_samples(X_train, y_train, self.random_state)
        sample_names = list(diverse_samples.keys())

        for model_name, model in self.base_models.items():
            for sample_name in sample_names:
                X_sample, y_sample = diverse_samples[sample_name]
                model.fit(X_sample, y_sample)

                predictions = model.predict(X_train)
                confidence_metrics = calculate_model_confidence_regression(
                    y_train, predictions, model_name
                )

                model_key = f"{model_name}_{sample_name}"
                self.model_confidences[model_key] = {
                    "model": model,
                    "confidence": confidence_metrics,
                }
                combined_conf = confidence_metrics["combined_confidence"]
                print(f"Обучена {model_key:<25} | Уверенность: {combined_conf:.3f}")
        print("\n Все модели успешно обучены!")

    # Предсказываем, используя только самые уверенные модели
    def predict(self, X, top_n=2):
        predictions_dict = {}
        confidence_scores = {}

        for model_key, model_info in self.model_confidences.items():
            model = model_info["model"]
            confidence = model_info["confidence"]["combined_confidence"]

            prediction = model.predict(X)
            predictions_dict[model_key] = prediction
            confidence_scores[model_key] = confidence

        # Выбираем топ-N моделей по уверенности
        top_models = sorted(
            confidence_scores.items(), key=lambda x: x[1], reverse=True
        )[:top_n]

        top_predictions = [predictions_dict[name] for name, conf in top_models]
        top_weights = [conf for name, conf in top_models]

        return np.average(top_predictions, axis=0, weights=top_weights)

    # Формирует таблицу с метриками уверенности моделей
    def get_confidence_table(self):
        table_records = []

        for model_key, model_info in self.model_confidences.items():
            model_name, sample_name = model_key.split("_", 1)
            confidence_data = model_info["confidence"]

            table_records.append(
                {
                    "Модель": model_name,
                    "Выборка": sample_name,
                    "R²": f"{confidence_data['r2']:.3f}",
                    "MAE": f"{confidence_data['mae']:.3f}",
                    "MSE": f"{confidence_data['mse']:.3f}",
                    "±1 балл": f"{confidence_data['precision_confidence']:.3f}",
                    "Итоговая уверенность": f"{confidence_data['combined_confidence']:.3f}",
                }
            )

        result_df = pd.DataFrame(table_records)
        result_df = result_df.sort_values("Итоговая уверенность", ascending=False)

        return result_df

    # Выводит форматированную таблицу с результатами моделей
    def print_confidence_table(self):
        confidence_df = self.get_confidence_table()

        console = Console()

        table = Table(
            title="ТАБЛИЦА УВЕРЕННОСТИ МОДЕЛЕЙ",
            box=box.ROUNDED,
            show_header=True,
            header_style="bold",
            title_style="bold",
            width=90,
        )

        table.add_column("Модель", justify="left", width=12)
        table.add_column("Выборка", justify="left", width=18)
        table.add_column("R²", justify="center", width=6)
        table.add_column("MAE", justify="center", width=6)
        table.add_column("MSE", justify="center", width=8)
        table.add_column("±1 балл", justify="center", width=8)
        table.add_column("Уверенность", justify="center", width=12)

        # Добавляем строки
        for _, row in confidence_df.iterrows():
            confidence_value = float(row["Итоговая уверенность"])

            table.add_row(
                row["Модель"],
                row["Выборка"],
                row["R²"],
                row["MAE"],
                row["MSE"],
                row["±1 балл"],
                row["Итоговая уверенность"],
            )

        console.print()
        console.print(table)

        # Вывод лучшей модели
        best_model_row = confidence_df.iloc[0]
        console.print(
            f"\nЛУЧШАЯ МОДЕЛЬ: {best_model_row['Модель']} "
            f"на выборке {best_model_row['Выборка']} "
            f"с уверенностью {best_model_row['Итоговая уверенность']}"
        )

In [177]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

model = BlendingModel(base_models)
model.fit(X_train, y_train)

model.print_confidence_table()

Обучена linear_full_70            | Уверенность: 0.534
Обучена linear_random_60          | Уверенность: 0.520
Обучена linear_balanced_sex       | Уверенность: 0.531
Обучена linear_balanced_education | Уверенность: 0.447
Обучена ridge_full_70             | Уверенность: 0.536
Обучена ridge_random_60           | Уверенность: 0.522
Обучена ridge_balanced_sex        | Уверенность: 0.533
Обучена ridge_balanced_education  | Уверенность: 0.447
Обучена lasso_full_70             | Уверенность: 0.517
Обучена lasso_random_60           | Уверенность: 0.495
Обучена lasso_balanced_sex        | Уверенность: 0.518
Обучена lasso_balanced_education  | Уверенность: 0.408
Обучена decision_tree_full_70     | Уверенность: 0.511
Обучена decision_tree_random_60   | Уверенность: 0.489
Обучена decision_tree_balanced_sex | Уверенность: 0.460
Обучена decision_tree_balanced_education | Уверенность: 0.487
Обучена knn_full_70               | Уверенность: 0.408
Обучена knn_random_60             | Уверенность: 0.367
Об

                               ТАБЛИЦА УВЕРЕННОСТИ МОДЕЛЕЙ                                
╭──────────────┬────────────────────┬────────┬────────┬─────────┬──────────┬─────────────╮
│ Модель       │ Выборка            │   R²   │  MAE   │   MSE   │ ±1 балл  │ Уверенность │
├──────────────┼────────────────────┼────────┼────────┼─────────┼──────────┼─────────────┤
│ ridge        │ full_70            │ 0.391  │ 1.856  │  6.075  │  0.366   │    0.536    │
│ linear       │ full_70            │ 0.381  │ 1.865  │  6.170  │  0.375   │    0.534    │
│ ridge        │ balanced_sex       │ 0.391  │ 1.853  │  6.072  │  0.353   │    0.533    │
│ linear       │ balanced_sex       │ 0.385  │ 1.863  │  6.129  │  0.356   │    0.531    │
│ ridge        │ random_60          │ 0.375  │ 1.909  │  6.230  │  0.344   │    0.522    │
│ linear       │ random_60          │ 0.375  │ 1.913  │  6.229  │  0.338   │    0.520    │
│ lasso        │ balanced_sex       │ 0.337  │ 1.872  │  6.608  │  0.375   │    0.518    │
│ lasso        │ full_70            │ 0.339  │ 1.893  │  6.595  │  0.372   │    0.517    │
│ decision     │ tree_full_70       │ 0.207  │ 1.880  │  7.905  │  0.530   │    0.511    │
│ lasso        │ random_60          │ 0.302  │ 1.957  │  6.961  │  0.347   │    0.495    │
│ decision     │ tree_random_60     │ 0.145  │ 1.927  │  8.521  │  0.536   │    0.489    │
│ decision     │ tree_balanced_edu… │ 0.188  │ 1.972  │  8.091  │  0.476   │    0.487    │
│ decision     │ tree_balanced_sex  │ 0.096  │ 2.000  │  9.009  │  0.505   │    0.460    │
│ ridge        │ balanced_education │ 0.149  │ 1.936  │  8.481  │  0.366   │    0.447    │
│ linear       │ balanced_education │ 0.149  │ 1.934  │  8.486  │  0.366   │    0.447    │
│ lasso        │ balanced_education │ 0.062  │ 2.038  │  9.353  │  0.350   │    0.408    │
│ knn          │ full_70            │ 0.000  │ 2.183  │ 10.726  │  0.498   │    0.408    │
│ knn          │ balanced_education │ 0.016  │ 2.230  │  9.808  │  0.451   │    0.407    │
│ knn          │ balanced_sex       │ 0.000  │ 2.161  │ 11.524  │  0.514   │    0.405    │
│ knn          │ random_60          │ 0.000  │ 2.394  │ 13.044  │  0.467   │    0.367    │
╰──────────────┴────────────────────┴────────┴────────┴─────────┴──────────┴─────────────╯

ЛУЧШАЯ МОДЕЛЬ: ridge на выборке full_70 с уверенностью 0.536

## Предсказание BlendingModel

In [178]:
result = model.predict(X_test, top_n=3)

mse = mean_squared_error(y_test, result)
mae = mean_absolute_error(y_test, result)
r2 = r2_score(y_test, result)

within_1_point = np.mean(np.abs(y_test - result) <= 1.0) * 100

print("\n" + "=" * 40)
print(f"МОДЕЛЬ: BlendingModel (Top-3)")
print("-" * 40)
print(f"MSE:      {mse:8.2f}")
print(f"MAE:      {mae:8.2f}")
print(f"R²:       {r2:8.3f}")
print(f"±1 балл:  {within_1_point:6.1f}%")
print("-" * 40)

target_achieved = mse < 10
status = "ЦЕЛЬ ДОСТИГНУТА" if target_achieved else "ЦЕЛЬ НЕДОСТИГНУТА"

print(status)
print("=" * 40)


МОДЕЛЬ: BlendingModel (Top-3)
----------------------------------------
MSE:          7.63
MAE:          1.94
R²:          0.272
±1 балл:    39.4%
----------------------------------------
ЦЕЛЬ ДОСТИГНУТА


## 4 Постройте решение на основе подхода Stacking

Правила:
- Реализуйте пайплайн обучения и предсказания (например, sklearn.pipeline или класс)
- Проведите оптимизацию пайплайна
- Оцените вклад каждого базового алгоритма в итоговое предсказание
- Цель: метрика MSE на тесте меньше 10

In [179]:
class StackingModel:
    def __init__(self, base_models, meta_model=None, random_seed=42):
        self.base_models = base_models
        self.meta_model = meta_model if meta_model is not None else LinearRegression()
        self.random_seed = random_seed
        self.fitted_base_models = {}

    def fit(self, features, target):
        X_train, X_validation, y_train, y_validation = train_test_split(
            features, target, test_size=0.3, random_state=self.random_seed
        )

        base_model_predictions = []

        for model_name, model_instance in self.base_models.items():
            model_instance.fit(X_train, y_train)
            self.fitted_base_models[model_name] = model_instance

            predictions = model_instance.predict(X_validation)
            base_model_predictions.append(predictions)

        meta_features = np.column_stack(base_model_predictions)

        self.meta_model.fit(meta_features, y_validation)

        return self

    def predict(self, features):
        base_predictions = []
        for model_name, fitted_model in self.fitted_base_models.items():
            prediction = fitted_model.predict(features)
            base_predictions.append(prediction)

        meta_features = np.column_stack(base_predictions)

        return self.meta_model.predict(meta_features)

In [180]:
X = data.drop("G3", axis=1)
y = data["G3"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [181]:
stacking_model = StackingModel(base_models)
stacking_model.fit(X_train, y_train)

y_pred = stacking_model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n" + "=" * 40)
print("РЕЗУЛЬТАТЫ БАЗОВОГО STACKING:")
print("-" * 40)
print(f"MSE:      {mse:8.2f}")
print(f"R²:       {r2:8.3f}")
print("-" * 40)

target_achieved = mse < 10
status = "ЦЕЛЬ ДОСТИГНУТА" if target_achieved else "ЦЕЛЬ НЕДОСТИГНУТА"

print(status)
print("=" * 40)


РЕЗУЛЬТАТЫ БАЗОВОГО STACKING:
----------------------------------------
MSE:          8.89
R²:          0.240
----------------------------------------
ЦЕЛЬ ДОСТИГНУТА


## Оценим вклад моделей

In [182]:
# Анализирует влияние каждой базовой модели на итоговый прогноз
def evaluate_model_contributions(stacking_model, X_test, y_test):
    print("\nАНАЛИЗ ВКЛАДА БАЗОВЫХ МОДЕЛЕЙ")
    print("─" * 45)

    model_performance = {}

    for model_name, model_instance in stacking_model.fitted_base_models.items():
        predictions = model_instance.predict(X_test)
        model_mse = mean_squared_error(y_test, predictions)
        model_performance[model_name] = model_mse
        print(f"{model_name:<20} → MSE: {model_mse:>6.2f}")

    # Анализ коэффициентов мета-модели
    if hasattr(stacking_model.meta_model, "coef_"):
        print("\nКОЭФФИЦИЕНТЫ МЕТА-МОДЕЛИ:")
        for model_name, coefficient in zip(
            stacking_model.fitted_base_models.keys(), stacking_model.meta_model.coef_
        ):
            print(f"{model_name:<20} → Коэффициент: {coefficient:>7.3f}")

    return model_performance


model_contributions = evaluate_model_contributions(stacking_model, X_test, y_test)


АНАЛИЗ ВКЛАДА БАЗОВЫХ МОДЕЛЕЙ
─────────────────────────────────────────────
linear               → MSE:   6.87
ridge                → MSE:   6.88
lasso                → MSE:   7.51
decision_tree        → MSE:   9.76
knn                  → MSE:  13.95

КОЭФФИЦИЕНТЫ МЕТА-МОДЕЛИ:
linear               → Коэффициент:  -7.089
ridge                → Коэффициент:   7.815
lasso                → Коэффициент:  -0.398
decision_tree        → Коэффициент:   0.194
knn                  → Коэффициент:   0.258


## Оптимизация

In [183]:
# Подбор оптимальной комбинации моделей для стекинга путем тестирования различных конфигураций базовых моделей и мета-моделей.
def optimize_stacking(X_train, y_train, X_test, y_test):

    # Тестовые конфигурации моделей для экспериментов
    test_configurations = [
        {
            "name": "Базовый набор",
            "base_models": {
                "linear_regression": LinearRegression(),
                "ridge_regression": Ridge(alpha=1.0),
                "random_forest": RandomForestRegressor(
                    n_estimators=50, random_state=42
                ),
            },
            "meta_model": LinearRegression(),
        },
        {
            "name": "С Lasso и KNN",
            "base_models": {
                "lasso_regression": Lasso(alpha=0.1),
                "random_forest": RandomForestRegressor(
                    n_estimators=100, random_state=42
                ),
                "knn": KNeighborsRegressor(n_neighbors=5),
            },
            "meta_model": Ridge(alpha=0.5),
        },
        {
            "name": "Расширенный набор",
            "base_models": {
                "linear_regression": LinearRegression(),
                "lasso_regression": Lasso(alpha=0.05),
                "random_forest": RandomForestRegressor(
                    n_estimators=100, random_state=42
                ),
                "knn": KNeighborsRegressor(n_neighbors=10),
            },
            "meta_model": Ridge(alpha=0.1),
        },
    ]

    best_performance = float("inf")
    optimal_config = None
    optimal_model = None

    print("\nПОДБОР ОПТИМАЛЬНОЙ КОНФИГУРАЦИИ STACKING")
    print("=" * 50)

    for config in test_configurations:
        # Создаем и обучаем стекинг-модель
        stacking_model = StackingModel(config["base_models"], config["meta_model"])
        stacking_model.fit(X_train, y_train)

        # Оцениваем качество
        predictions = stacking_model.predict(X_test)
        current_mse = mean_squared_error(y_test, predictions)

        print(f"  {config['name']:<25} → MSE: {current_mse:6.2f}")

        # Сохраняем лучший результат
        if current_mse < best_performance:
            best_performance = current_mse
            optimal_config = config
            optimal_model = stacking_model

    print("=" * 50)
    print(f"ЛУЧШАЯ КОНФИГУРАЦИЯ: {optimal_config['name']}")
    print(f"ЛУЧШИЙ MSE: {best_performance:.2f}")

    return optimal_model, optimal_config, best_performance


# Запуск оптимизации
best_stacking, best_config, best_mse = optimize_stacking(
    X_train, y_train, X_test, y_test
)


ПОДБОР ОПТИМАЛЬНОЙ КОНФИГУРАЦИИ STACKING
  Базовый набор             → MSE:   7.81
  С Lasso и KNN             → MSE:   7.79
  Расширенный набор         → MSE:   7.79
ЛУЧШАЯ КОНФИГУРАЦИЯ: С Lasso и KNN
ЛУЧШИЙ MSE: 7.79


In [184]:
print("\n" + "=" * 50)
print("ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ ОПТИМИЗАЦИИ")
print("=" * 50)

print(f"\nЛУЧШАЯ КОНФИГУРАЦИЯ: {best_config['name']}")
print(f"Базовые модели: {list(best_config['base_models'].keys())}")
print(f"Мета-модель: {type(best_config['meta_model']).__name__}")
print(f"MSE на тестовой выборке: {best_mse:.2f}")

target_status = "ДОСТИГНУТА" if best_mse < 10 else "НЕ ДОСТИГНУТА"
print(f"Цель (MSE < 10): {target_status}")

print("\n" + "-" * 50)
print("АНАЛИЗ ВКЛАДА МОДЕЛЕЙ В ЛУЧШЕЙ КОНФИГУРАЦИИ")
print("-" * 50)

evaluate_model_contributions(best_stacking, X_test, y_test)


ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ ОПТИМИЗАЦИИ

ЛУЧШАЯ КОНФИГУРАЦИЯ: С Lasso и KNN
Базовые модели: ['lasso_regression', 'random_forest', 'knn']
Мета-модель: Ridge
MSE на тестовой выборке: 7.79
Цель (MSE < 10): ДОСТИГНУТА

--------------------------------------------------
АНАЛИЗ ВКЛАДА МОДЕЛЕЙ В ЛУЧШЕЙ КОНФИГУРАЦИИ
--------------------------------------------------

АНАЛИЗ ВКЛАДА БАЗОВЫХ МОДЕЛЕЙ
─────────────────────────────────────────────
lasso_regression     → MSE:   7.51
random_forest        → MSE:   7.47
knn                  → MSE:  10.85

КОЭФФИЦИЕНТЫ МЕТА-МОДЕЛИ:
lasso_regression     → Коэффициент:   0.165
random_forest        → Коэффициент:   0.747
knn                  → Коэффициент:  -0.295


{'lasso_regression': 7.5116546289366735,
 'random_forest': 7.474774725274726,
 'knn': 10.852307692307692}

## * Доп задание (не обязательно, но решение будет поощряться)

Правила:
- Постройте несколько сильных алгоритмов разного класса (это может быть бустинг, нейросеть, ансамбль слабых алгоритмов, алгоритм на статистике, что придумаете)
- Реализуйте "управляющий" алгоритм, который на основе входных данных будет выбирать, какой из  сильных алгоритмов запустить (не на основе их работы, а именно на основе данных)